In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt
import seaborn as sns


In [31]:
anime = pd.read_csv("Anime.csv")
ratings = pd.read_csv("Rating.csv")


In [32]:
print(ratings.head())

   user_id  anime_id  rating
0        1        20      -1
1        1        24      -1
2        1        79      -1
3        1       226      -1
4        1       241      -1


In [33]:
ratings = ratings.dropna(subset=["user_id", "anime_id", "rating"])
ratings["rating"] = ratings["rating"].astype(float)


In [34]:
from scipy.sparse import csr_matrix

user_ids = ratings["user_id"].astype("category").cat.codes
anime_ids = ratings["anime_id"].astype("category").cat.codes

user_anime_matrix = csr_matrix(
    (ratings["rating"], (user_ids, anime_ids))
)


In [35]:
from sklearn.preprocessing import normalize

user_anime_norm = normalize(user_anime_matrix, axis=1)


In [37]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def get_top_k_similar_users(user_index, matrix, k=10):
    similarities = cosine_similarity(
        matrix[user_index],
        matrix
    ).flatten()

    similarities[user_index] = 0  # remove self
    top_k_users = np.argsort(similarities)[-k:]

    return top_k_users, similarities[top_k_users]


In [65]:
def recommend_anime_with_names(
    user_id_original,
    ratings_df,
    anime_df,
    matrix,
    k_users=10,
    n_recommendations=5
):
    # map user_id → index
    user_map = ratings_df["user_id"].astype("category")
    user_index = user_map.cat.categories.get_loc(user_id_original)

    # get similar users
    similar_users, similarities = get_top_k_similar_users(
        user_index, matrix, k=k_users
    )

    # anime already watched
    watched_anime = ratings_df[
        ratings_df["user_id"] == user_id_original
    ]["anime_id"].unique()

    # similar users original IDs
    similar_users_ids = user_map.cat.categories[similar_users]

    # candidate anime
    candidates = ratings_df[
        ratings_df["user_id"].isin(similar_users_ids)
        & ~ratings_df["anime_id"].isin(watched_anime)
    ]

    # average rating score
    anime_scores = (
        candidates.groupby("anime_id")["rating"]
        .mean()
        .reset_index()
        .sort_values("rating", ascending=False)
        .head(n_recommendations)
    )

    # merge anime names (only needed column)
    final_recommendations = anime_scores.merge(
        anime_df[["anime_id", "name"]],   # 👈 IMPORTANT
        on="anime_id",
        how="left"
    )

    # final clean output
    final_recommendations = final_recommendations.rename(
        columns={"name": "anime_name"}
    )

    return final_recommendations[
        ["anime_id", "rating", "anime_name"]
    ]


In [ ]:
final_output = recommend_anime_with_names(
    user_id_original=user_id_to_test,
    ratings_df=ratings,
    anime_df=anime,
    matrix=user_anime_norm,
    k_users=10,
    n_recommendations=5
)

print(final_output)


   anime_id  rating            anime_name
0       205    10.0      Samurai Champloo
1       813    10.0         Dragon Ball Z
2     12531    10.0  Sakamichi no Apollon
3      1292    10.0          Afro Samurai
4     16498     9.0    Shingeki no Kyojin
